# **Creating a Custom Template**

### **Objective:**  
To learn how to create custom prompt templates using Pydantic models and template classes, enabling dynamic prompt generation with input validation and structured template management for scalable AI workflows.

---

### **Note:**  
- Before running any demo, ensure that the **requirements.txt** file is installed. This file contains all the required dependencies for **all demos and guided practices under Building LLM Applications**.

- If the dependencies were already installed earlier (after creating the virtual environment), there is no need to install them again. You can directly proceed with running the demo.

- Refer to Lesson_01
**Demo_01_Zero_Shot_Prompting.ipynb** Step 1 
for creating a virtual environment and installing the requirements.txt 

- Ensure you select the right kernel **Python (myenv)** while running the demos
---

### **Steps to perform:**
1. Set up the environment  
2. Define the prompt template  
3. Create a custom prompt template class  

---


### __Step 1: Set up the environment__

- Import required modules such as StringPromptTemplate from the langchain.prompts library and BaseModel and validator from Pydantic library.



In [ ]:

from pydantic import BaseModel, validator, ValidationError
from typing import List
from openai import OpenAI
from pydantic.config import ConfigDict

client = OpenAI()


- Pydantic helps validate and structure data, ensuring inputs are in the correct format and automatically converting types if needed. It is widely used for validating user input, API requests, and managing configurations effectively.

### __Step 2: Define the prompt template__
- Define a constant string that outlines the structure of the prompt for generating book summaries

In [2]:
PROMPT = """\
Given the book title, generate a brief summary of the book.
Book Title: {book_title}
Summary:
"""


- The prompt includes a placeholder {book_title} that allows dynamic content insertion during execution.

### __Step 3: Create a custom prompt template class__
- Create a Pydantic model to validate data and integrate dynamic content into templates

In [ ]:

class StringPromptTemplate:
    def __init__(self, template: str):
        self.template = template

    def format(self, **kwargs) -> str:
        return self.template.format(**kwargs)



- Define a BookSummarizerPrompt model to use the above template for structured prompt generation

In [ ]:

class BookSummarizerPrompt(BaseModel):
    book_title: str
    template: StringPromptTemplate

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def create_prompt(self) -> str:
        return self.template.format(book_title=self.book_title)


- Create an instance of the template and dynamically generate a formatted prompt

In [ ]:
# Define the template
template = StringPromptTemplate(
    """
Given the book title, generate a brief summary of the book.
Book Title: {book_title}
Summary:
"""
)

# Create instance of BookSummarizerPrompt
book_summarizer_prompt = BookSummarizerPrompt(book_title="The Great Gatsby", template=template)

# Generate prompt
formatted_prompt = book_summarizer_prompt.create_prompt()
print("Generated Prompt:")
print(formatted_prompt)


- Define a function to send the formatted prompt to the OpenAI API and get the AI-generated response

In [ ]:

# Define OpenAI completion function
def get_completion(prompt: str, model="gpt-3.5-turbo") -> str:
    try:
        messages = [{"role": "user", "content": prompt}]
        
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.1,
            top_p=0.8,
            max_tokens=512
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"



- Generate the summary by passing the custom prompt to the OpenAI model

In [ ]:
# Get AI-generated summary
response = get_completion(formatted_prompt)
print("\nAI Response:")
print(response)


In [ ]:
### Example: Recipe generator using custom template class

# Define a template class to manage dynamic text
class RecipePromptTemplate:
    def __init__(self, template: str):
        self.template = template

    def format(self, **kwargs):
        return self.template.format(**kwargs)

# Define a pydantic model for recipe inputs
class RecipeGenerator(BaseModel):
    recipe_name: str
    ingredients: List[str]
    template: RecipePromptTemplate

    def create_prompt(self):
        # Join ingredients into a single string for the template
        ingredients_text = ", ".join(self.ingredients)
        return self.template.format(recipe_name=self.recipe_name, ingredients=ingredients_text)

    model_config = ConfigDict(arbitrary_types_allowed=True)

# Define the prompt template
recipe_template = RecipePromptTemplate(
    """
    Generate a detailed recipe for the following dish:
    Recipe Name: {recipe_name}
    Ingredients: {ingredients}
    Instructions:
    """
)

# Create an instance of the recipe generator
recipe_data = {
    "recipe_name": "Pizza",
    "ingredients": ["Refined Flour", "eggs", "parmesan cheese", "Mushrooms", "black pepper"],
    "template": recipe_template,
}

recipe_generator = RecipeGenerator(**recipe_data)

# Generate the prompt
prompt = recipe_generator.create_prompt()
print("Prompt:")
print(prompt)
response = get_completion(recipe_generator.create_prompt())
print("AI Response:")
print(response)

Prompt:

    Generate a detailed recipe for the following dish:
    Recipe Name: Pizza
    Ingredients: Refined Flour, eggs, parmesan cheese, Mushrooms, black pepper
    Instructions:
    
AI Response:
1. Preheat your oven to 450°F (230°C).

2. In a mixing bowl, combine 2 cups of refined flour, 2 eggs, and 1/2 cup of grated parmesan cheese. Mix well until a dough forms.

3. On a floured surface, roll out the dough into a round shape, about 1/4 inch thick.

4. Transfer the dough to a pizza pan or baking sheet lined with parchment paper.

5. Spread a thin layer of tomato sauce over the dough, leaving a small border around the edges.

6. Top the pizza with sliced mushrooms and sprinkle with black pepper to taste.

7. Bake the pizza in the preheated oven for 15-20 minutes, or until the crust is golden brown and the toppings are cooked.

8. Remove the pizza from the oven and let it cool for a few minutes before slicing and serving.

9. Enjoy your homemade pizza with a side salad or your fav

### __Conclusion__
By following these steps, you have learned how to create custom prompt templates using Pydantic models and custom classes. This approach helps validate inputs, dynamically format text, and build reusable, structured templates for various AI-driven workflows such as book summaries, recipes, and content generation.

---